In [4]:
import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import train_test_split

In [5]:

base = pd.read_csv("base_classificacao_beat_mes.csv")

base["MES_REF"] = pd.to_datetime(base["MES_REF"], errors="coerce")
base = base.dropna(subset=["MES_REF"])

# ============================================================
# 2. Definir features e alvo
# ============================================================

features_modelo = [
    "total_acidentes_ult_6m",
    "media_acidentes_mensal_ult_6m",

    "perc_acidentes_noite_ult_6m",
    "perc_acidentes_fim_semana_ult_6m",

    "perc_iluminacao_risco_ult_6m",
    "perc_pista_risco_ult_6m",
    "perc_defeito_via_ult_6m",
    "perc_dispositivo_problema_ult_6m",
    "perc_clima_risco_ult_6m",

    "velocidade_media_ult_6m",
    "num_units_medio_ult_6m",

    "mes_previsao"
]

target = "prioridade_futura"

# Garantir que não há nulos nas features
base[features_modelo] = base[features_modelo].fillna(0)

X = base[features_modelo]
y = base[target]




In [9]:

# ============================================================
# 3. Separação temporal treino/teste
# ============================================================
# Importante:
# Não usar split aleatório como principal.
# O modelo precisa aprender com o passado e ser testado no futuro.

data_corte = pd.Timestamp("2024-12-01")

treino = base[base["MES_REF"] <= data_corte].copy()
teste = base[base["MES_REF"] > data_corte].copy()

# Caso a base não tenha dados suficientes nesse corte, usa 70% temporal
if len(treino) == 0 or len(teste) == 0:
    meses = sorted(base["MES_REF"].unique())
    corte_auto = meses[int(len(meses) * 0.7)]

    treino = base[base["MES_REF"] <= corte_auto].copy()
    teste = base[base["MES_REF"] > corte_auto].copy()

    print(f"Corte automático usado: {corte_auto}")

X_train = treino[features_modelo]
y_train = treino[target]

X_test = teste[features_modelo]
y_test = teste[target]

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

print("\nDistribuição treino:")
print(y_train.value_counts(normalize=True))

print("\nDistribuição teste:")
print(y_test.value_counts(normalize=True))

Treino: (22032, 12)
Teste: (4352, 12)

Distribuição treino:
prioridade_futura
BAIXA    0.500590
MEDIA    0.293981
ALTA     0.205428
Name: proportion, dtype: float64

Distribuição teste:
prioridade_futura
BAIXA    0.514706
MEDIA    0.285846
ALTA     0.199449
Name: proportion, dtype: float64


In [12]:
X.columns

Index(['total_acidentes_ult_6m', 'media_acidentes_mensal_ult_6m',
       'perc_acidentes_noite_ult_6m', 'perc_acidentes_fim_semana_ult_6m',
       'perc_iluminacao_risco_ult_6m', 'perc_pista_risco_ult_6m',
       'perc_defeito_via_ult_6m', 'perc_dispositivo_problema_ult_6m',
       'perc_clima_risco_ult_6m', 'velocidade_media_ult_6m',
       'num_units_medio_ult_6m', 'mes_previsao'],
      dtype='str')

In [10]:
# ============================================================
# 4. Treinar modelo
# ============================================================


modelo = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

modelo.fit(X_train, y_train)

# ============================================================
# 5. Avaliar modelo
# ============================================================

y_pred = modelo.predict(X_test)

print("\nAcurácia:")
print(accuracy_score(y_test, y_pred))

print("\nMatriz de confusão:")
print(confusion_matrix(y_test, y_pred, labels=["BAIXA", "MEDIA", "ALTA"]))

print("\nRelatório de classificação:")
print(classification_report(y_test, y_pred))

# Métricas específicas da classe ALTA
precision, recall, f1, support = precision_recall_fscore_support(
    y_test,
    y_pred,
    labels=["ALTA"],
    zero_division=0
)

print("\nMétricas da classe ALTA:")
print(f"Precision ALTA: {precision[0]:.4f}")
print(f"Recall ALTA: {recall[0]:.4f}")
print(f"F1 ALTA: {f1[0]:.4f}")
print(f"Suporte ALTA: {support[0]}")

# ============================================================
# 6. Probabilidades previstas
# ============================================================

probas = modelo.predict_proba(X_test)
classes = modelo.classes_

teste_resultado = teste.copy()
teste_resultado["prioridade_prevista"] = y_pred

for i, classe in enumerate(classes):
    teste_resultado[f"prob_{classe.lower()}"] = probas[:, i]

# Salvar resultados do teste
teste_resultado.to_csv("predicoes_teste.csv", index=False)

# ============================================================
# 7. Importância das variáveis
# ============================================================

importancias = pd.DataFrame({
    "feature": features_modelo,
    "importance": modelo.feature_importances_
}).sort_values("importance", ascending=False)

importancias.to_csv("importancia_features.csv", index=False)

print("\nImportância das variáveis:")
print(importancias)

# ============================================================
# 8. Salvar modelo e configurações
# ============================================================

artefatos = {
    "modelo": modelo,
    "features": features_modelo,
    "classes": list(modelo.classes_)
}

joblib.dump(artefatos, "modelo_prioridade_beat.pkl")

print("\nModelo salvo em: modelo_prioridade_beat.pkl")
print("Predições de teste salvas em: predicoes_teste.csv")
print("Importância das features salva em: importancia_features.csv")


Acurácia:
0.7828584558823529

Matriz de confusão:
[[1926  307    7]
 [ 220  800  224]
 [  11  176  681]]

Relatório de classificação:
              precision    recall  f1-score   support

        ALTA       0.75      0.78      0.77       868
       BAIXA       0.89      0.86      0.88      2240
       MEDIA       0.62      0.64      0.63      1244

    accuracy                           0.78      4352
   macro avg       0.75      0.76      0.76      4352
weighted avg       0.79      0.78      0.78      4352


Métricas da classe ALTA:
Precision ALTA: 0.7467
Recall ALTA: 0.7846
F1 ALTA: 0.7652
Suporte ALTA: 868

Importância das variáveis:
                             feature  importance
1      media_acidentes_mensal_ult_6m    0.340788
0             total_acidentes_ult_6m    0.331703
11                      mes_previsao    0.062103
5            perc_pista_risco_ult_6m    0.040474
7   perc_dispositivo_problema_ult_6m    0.032837
9            velocidade_media_ult_6m    0.030281
6         